# Train and publish the release classifier

This trains the CiteAudit classifier that the `citeaudit` package downloads and publishes it on
the Hugging Face Hub. It is the NLI model of `results/nli_classifier_v1.md`, with the same data and
settings and seed 2: of the three seeds there, seed 2 scored best on dev, and dev is what chose
it. It trains on train, chooses the epoch on dev, then scores test (about 40 minutes).

**Before running:**
1. Make a Hugging Face write token: huggingface.co → Settings → Access Tokens → Create new token
   → type **Write**.
2. In Colab, click the **key icon** in the left sidebar (Secrets) → **Add new secret**. Name it
   `HF_TOKEN`, paste the token as the value, and switch on **Notebook access**.
3. Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow Google Drive access.

The model goes to `<your username>/citeaudit-nli` on the Hub, and a copy is saved to
`MyDrive/CiteAudit_runs/release`. Without the token, the model is only saved to Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/release'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19" sentencepiece protobuf huggingface_hub

import subprocess, json, statistics
def run(cmd):
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS',
             'Loading weights', 'Warning', 'warn(')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

T = 'data_train/target'
def train(name, seed, extra):
    out = f'{RES}/{name}'
    if os.path.exists(f'{out}/config.json'):
        print(f'{name}: already done'); return
    run(f'python scripts/train_nli.py --train {T}/ours_20r_train_cv --dev {T}/ours_20r '
        f'--epochs 3 --batch 2 --accum 4 --seed {seed} --amp '
        f'--out checkpoints/{name} --pred-dir {out}.part {extra}')
    run(f'rm -rf checkpoints/{name}')
    os.rename(f'{out}.part', out)

LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(rows):
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1
read = lambda p: [json.loads(l) for l in open(p)]
COLS = ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']

## Train (seed 2), score test, save the model

In [ ]:
MODEL_DIR = f'{RES}/citeaudit-nli'
if not os.path.exists(f'{MODEL_DIR}/model.safetensors'):
    run(f'python scripts/train_nli.py --train {T}/ours_20r_train_cv --dev {T}/ours_20r '
        f'--epochs 3 --batch 2 --accum 4 --seed 2 --amp --out checkpoints/release '
        f'--pred-dir {RES}/preds --predict test={T}/ours_20r_test --save-model {MODEL_DIR}')
test = scores(read(f'{RES}/preds/ours_20r_test.jsonl'))
print('test:', dict(zip(COLS, (round(x, 3) for x in test))))
json.dump(dict(zip(COLS, test)), open(f'{MODEL_DIR}/test_metrics.json', 'w'), indent=2)

## Publish to the Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception as e:
    token = None
    print('No HF_TOKEN secret found, so the model is saved to Drive only:', MODEL_DIR)
if token:
    api = HfApi(token=token)
    repo = f"{api.whoami()['name']}/citeaudit-nli"
    card = open('release/nli_model_card.md').read().replace('REPO_ID', repo)
    open(f'{MODEL_DIR}/README.md', 'w').write(card)
    api.create_repo(repo, exist_ok=True)
    api.upload_folder(folder_path=MODEL_DIR, repo_id=repo, commit_message='CiteAudit release classifier')
    print('published: https://huggingface.co/' + repo)

## Download

Zips the predictions and logs (not the model) to `release.zip`. Put it in the repository root and
tell Claude the Hub link printed above.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/release.zip release -x 'release/citeaudit-nli/*.safetensors'
from google.colab import files
files.download('/content/release.zip')